# Moment-DETR + QVHighlights Baseline

## 목적

자연어 문장형 질의를 입력했을 때 **영상에서 관련 구간의 시작/종료 timestamp를 찾는 Video Moment Retrieval 모델**이 실제로 동작하는지 확인한다.

이번 노트북의 1차 목표는 다음과 같다.

- Google Colab GPU 환경에서 Moment-DETR 실행
- QVHighlights annotation 구조 확인
- 공식 pretrained checkpoint로 예제 영상 inference
- 직접 수정한 문장형 query로 timestamp 예측
- 저장소에 포함된 baseline metric 확인

> **중요:** 이 노트북은 아직 우리 서비스 영상에 적용한 단계가 아니다.  
> 먼저 공식 Moment-DETR + QVHighlights 환경에서 baseline이 정상 동작하는지 검증한 기록이다.


## 1. 실행 환경 확인

Moment-DETR의 영상 feature 추출 및 inference를 위해 Colab GPU를 사용한다.  
이번 실행에서는 **Tesla T4**가 할당되었다.


In [ ]:
!nvidia-smi


Mon Sep 21 06:10:42 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   58C    P8             12W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

### PyTorch / CUDA 확인

현재 Colab의 PyTorch 버전과 CUDA 사용 가능 여부를 확인한다.

이번 실행 환경:
- PyTorch: `2.11.0+cu128`
- CUDA 사용 가능: `True`
- GPU: `Tesla T4`


In [ ]:
import torch

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None")


PyTorch: 2.11.0+cu128
CUDA available: True
GPU: Tesla T4


## 2. Moment-DETR 공식 코드 준비

공식 GitHub 저장소를 clone하고 프로젝트 디렉터리로 이동한다.

- Repository: `jayleicn/moment_detr`
- 이후 모든 코드는 `/content/moment_detr` 기준으로 실행한다.


In [ ]:
!git clone https://github.com/jayleicn/moment_detr.git
%cd /content/moment_detr
!ls


## 3. 필수 라이브러리 설치

Moment-DETR 실행과 video demo에 필요한 패키지를 설치한다.

`ffmpeg-python`, `ftfy`, `regex`는 `run_on_video` 예제 실행에 필요하다.


In [ ]:
!pip install -q tqdm easydict tensorboard tabulate scikit-learn pandas ffmpeg-python ftfy regex

import tqdm
import easydict
import pandas
import sklearn

print("필수 패키지 import 성공")


필수 패키지 import 성공


## 4. QVHighlights annotation 구조 확인

Moment-DETR 저장소의 `data/` 폴더에는 QVHighlights의 train / validation / test annotation이 포함되어 있다.

주요 파일:
- `highlight_train_release.jsonl` : 학습용
- `highlight_val_release.jsonl` : 검증용
- `highlight_test_release.jsonl` : 테스트용
- `highlight_test_with_gt.jsonl` : Ground Truth가 포함된 테스트 데이터

QVHighlights의 핵심 필드는 다음과 같다.

| 필드 | 의미 |
|---|---|
| `qid` | query ID |
| `query` | 자연어 검색 문장 |
| `vid` | 영상 ID |
| `duration` | 영상 길이(초) |
| `relevant_windows` | query와 관련 있는 정답 시간 구간 |
| `relevant_clip_ids` | 관련 clip ID |
| `saliency_scores` | clip별 중요도 annotation |

즉 학습의 핵심 관계는 **Query + Video Feature → Ground Truth Timestamp**이다.


In [ ]:
!ls data

import json

with open("data/highlight_train_release.jsonl", "r", encoding="utf-8") as f:
    sample = json.loads(f.readline())

print("Query:", sample["query"])
print("Video ID:", sample["vid"])
print("Duration:", sample["duration"])
print("Ground Truth Windows:", sample["relevant_windows"])


### 참고: QVHighlights pre-extracted feature

Moment-DETR 전체 validation/test 재평가에는 QVHighlights의 pre-extracted video feature가 필요하다.

이번 작업에서 공식 feature 다운로드 링크를 시도했으나 **404 응답으로 다운로드되지 않았다.**  
따라서 이번 노트북에서는 전체 데이터셋을 새로 평가하지 않고,

1. 저장소에 포함된 공식 example video
2. pretrained checkpoint
3. 저장소에 포함된 기존 evaluation 결과

를 이용해 baseline 동작을 우선 검증했다.


## 5. 최신 PyTorch 호환성 수정

현재 Colab은 최신 PyTorch를 사용하고 있으며, PyTorch 2.6 이후 `torch.load()`의 `weights_only` 기본 동작이 변경되어 기존 Moment-DETR checkpoint 로딩 시 오류가 발생했다.

공식 저장소에서 받은 checkpoint를 사용하므로 `run_on_video/model_utils.py`의 checkpoint 로딩 부분에 `weights_only=False`를 명시한다.

> 이 수정은 모델 구조를 변경하는 것이 아니라 **최신 PyTorch에서 기존 checkpoint를 읽기 위한 호환성 수정**이다.


In [ ]:
from pathlib import Path

model_utils_path = Path("/content/moment_detr/run_on_video/model_utils.py")
text = model_utils_path.read_text()

old = 'torch.load(ckpt_path, map_location="cpu")'
new = 'torch.load(ckpt_path, map_location="cpu", weights_only=False)'

if old in text:
    text = text.replace(old, new)
    model_utils_path.write_text(text)
    print("model_utils.py 호환성 수정 완료")
else:
    print("이미 수정되어 있거나 대상 코드가 변경되었습니다.")


## 6. 공식 pretrained demo 실행

먼저 공식 example video와 query를 그대로 사용해 모델이 정상 동작하는지 확인한다.

**Query**
> `Chef makes pizza and cuts it up.`

**Ground Truth**
> `106 ~ 122 sec`

이 단계의 목표는 모델을 수정하기 전에 **문장 → 영상 구간 timestamp 예측이 정상적으로 수행되는지** 확인하는 것이다.


In [ ]:
!PYTHONPATH=$PYTHONPATH:. python run_on_video/run.py


Build models...
Loading feature extractors...
Loading CLIP models
Loading trained Moment-DETR model...
Run prediction...
------------------------------idx0
>> query: Chef makes pizza and cuts it up.
>> video_path: run_on_video/example/RoripwjYFp8_60.0_210.0.mp4
>> GT moments: [[106, 122]]
>> Predicted moments ([start_in_seconds, end_in_seconds, score]): [[49.7024, 64.3844, 0.9899], [111.1009, 126.1592, 0.9787], [104.1799, 115.1005, 0.8507], [62.7127, 72.0642, 0.4094], [128.7344, 138.6824, 0.0351], [42.1975, 50.5584, 0.0146], [114.1488, 120.6217, 0.0031], [127.7778, 134.0186, 0.003], [34.3171, 40.7786, 0.0029], [14.56, 21.1612, 0.0004]]
>> GT saliency scores (only localized 2-sec clips): [[2, 3, 3], [2, 3, 3], [4, 3, 3], [3, 4, 3], [3, 4, 0], [3, 4, 0], [2, 4, 0], [2, 4, 0]]
>> Predicted saliency scores (for all 2-sec clip): [-0.7632, -0.7441, -0.5205, -0.8159, -0.6377, -0.8022, -0.7524, -0.79, -0.7065, -0.7285, -0.6118, -0.6064, -0.6436, -0.5894, -0.6895, -0.6621, -0.6353, -0.4941, -0.

### 공식 demo 결과 해석

Ground Truth는 `106 ~ 122초`이다.

예측 후보 중:
- `111.10 ~ 126.16초` / score `0.9787`
- `104.18 ~ 115.10초` / score `0.8507`

처럼 정답 구간과 크게 겹치는 후보가 생성되었다.

따라서 pretrained Moment-DETR이 **자연어 query를 입력받아 관련 영상 구간의 start/end timestamp 후보를 출력하는 것**을 확인했다.

> 최고 score 후보가 항상 Ground Truth와 가장 잘 겹치는 것은 아니므로, 이후 실제 서비스에서는 ranking·후처리·fine-tuning을 함께 검토해야 한다.


## 7. Custom Query 단일 예측 테스트

이번에는 같은 example video에 대해 query 문장을 직접 변경한다.

기존 query:
> `Chef makes pizza and cuts it up.`

변경 query:
> `A person cuts the pizza.`

원본 annotation 파일은 유지하고 `queries_custom.jsonl`을 별도로 만든다.


In [ ]:
import json
from pathlib import Path

src = Path("run_on_video/example/queries.jsonl")
dst = Path("run_on_video/example/queries_custom.jsonl")

with src.open("r", encoding="utf-8") as f:
    item = json.loads(f.readline())

item["query"] = "A person cuts the pizza."

with dst.open("w", encoding="utf-8") as f:
    f.write(json.dumps(item) + "\n")

print("Custom Query:", item["query"])
print("Ground Truth:", item["relevant_windows"])


### Custom query 파일 연결

`run_on_video/run.py`가 기본 `queries.jsonl` 대신 `queries_custom.jsonl`을 읽도록 변경한다.

이 수정은 **테스트 query 입력 파일 경로만 변경**하는 것이다.


In [ ]:
from pathlib import Path

run_path = Path("run_on_video/run.py")
text = run_path.read_text()

original = 'query_path = "run_on_video/example/queries.jsonl"'
custom = 'query_path = "run_on_video/example/queries_custom.jsonl"'

if original in text:
    text = text.replace(original, custom)
    run_path.write_text(text)
    print("Custom query 연결 완료")
elif custom in text:
    print("이미 Custom query가 연결되어 있습니다.")
else:
    print("query_path 위치를 확인해야 합니다.")


In [ ]:
!PYTHONPATH=$PYTHONPATH:. python run_on_video/run.py


Build models...
Loading feature extractors...
Loading CLIP models
Loading trained Moment-DETR model...
Run prediction...
------------------------------idx0
>> query: A person cuts the pizza.
>> video_path: run_on_video/example/RoripwjYFp8_60.0_210.0.mp4
>> GT moments: [[106, 122]]
>> Predicted moments ([start_in_seconds, end_in_seconds, score]): [[52.1608, 67.905, 0.9837], [110.6294, 123.1011, 0.9395], [69.1535, 82.3239, 0.7775], [97.1496, 106.9517, 0.2665], [139.7232, 149.0525, 0.0432], [125.8287, 134.1619, 0.0204], [108.4895, 116.7363, 0.006], [38.8625, 45.9226, 0.0012], [45.9046, 50.6759, 0.0011], [23.9219, 33.829, 0.0007]]
>> GT saliency scores (only localized 2-sec clips): [[2, 3, 3], [2, 3, 3], [4, 3, 3], [3, 4, 3], [3, 4, 0], [3, 4, 0], [2, 4, 0], [2, 4, 0]]
>> Predicted saliency scores (for all 2-sec clip): [-0.7803, -0.8267, -0.6865, -0.7983, -0.7695, -0.6567, -0.7915, -0.6934, -0.6279, -0.7773, -0.6401, -0.6836, -0.5596, -0.6465, -0.6523, -0.5737, -0.4385, -0.4485, -0.5122, -

### Custom Query 결과 해석

**Query**
> `A person cuts the pizza.`

**Ground Truth**
> `106 ~ 122 sec`

예측 후보 중:
- `110.63 ~ 123.10초` / score `0.9395`

가 Ground Truth와 크게 겹쳤다.

즉, 공식 query를 그대로 실행한 것뿐 아니라 **직접 작성한 자연어 문장으로도 Moment-DETR이 새로운 timestamp 후보를 계산하는 것**을 확인했다.


## 8. 저장소의 Baseline Metric 확인

공식 QVHighlights feature 다운로드 링크 문제로 전체 validation/test를 현재 Colab에서 직접 재평가하지는 못했다.

대신 pretrained checkpoint 디렉터리에 포함된 기존 evaluation 결과 파일을 확인한다.

주요 지표:
- `R1@0.5`: 예측 1순위 구간이 Ground Truth와 IoU 0.5 이상 겹치는 비율
- `R1@0.7`: 더 엄격한 IoU 0.7 기준 Recall@1
- `mAP`: 여러 IoU threshold에서의 평균 precision 성능


In [ ]:
import json

metrics_path = "run_on_video/moment_detr_ckpt/inference_hl_val_test_code_preds_metrics.json"

with open(metrics_path, "r") as f:
    metrics = json.load(f)

brief = metrics["brief"]

print("=== Moment-DETR QVHighlights Baseline ===")
print(f"R1@0.5   : {brief['MR-full-R1@0.5']}")
print(f"R1@0.7   : {brief['MR-full-R1@0.7']}")
print(f"mAP      : {brief['MR-full-mAP']}")
print(f"mAP@0.5  : {brief['MR-full-mAP@0.5']}")
print(f"mAP@0.75 : {brief['MR-full-mAP@0.75']}")
print()
print(f"HL Fair mAP     : {brief['HL-min-Fair-mAP']}")
print(f"HL Good mAP     : {brief['HL-min-Good-mAP']}")
print(f"HL VeryGood mAP : {brief['HL-min-VeryGood-mAP']}")


=== Moment-DETR QVHighlights Baseline ===
R1@0.5   : 53.23
R1@0.7   : 34.0
mAP      : 30.58
mAP@0.5  : 54.8
mAP@0.75 : 29.02

HL Fair mAP     : 68.29
HL Good mAP     : 57.93
HL VeryGood mAP : 35.51


## 9. 현재까지 완료된 내용

### 완료
- [x] Colab GPU 환경 확인
- [x] Moment-DETR 공식 코드 실행
- [x] 필수 라이브러리 설치
- [x] QVHighlights annotation 구조 확인
- [x] 최신 PyTorch checkpoint 호환성 수정
- [x] 공식 pretrained demo inference 성공
- [x] 직접 작성한 custom query inference 성공
- [x] 저장소에 포함된 baseline metric 확인

### 아직 진행하지 않은 내용
- [ ] QVHighlights 전체 feature를 이용한 validation/test 직접 재평가
- [ ] QVHighlights fine-tuning
- [ ] 우리 프로젝트 영상의 Moment-DETR용 visual feature 생성
- [ ] 우리 영상 + 사용자 query inference
- [ ] 백엔드용 `video_id / start_time / end_time / score` 반환 함수 구현


## 10. 다음 개발 단계

현재까지는 **공식 데이터와 공식 pretrained model이 정상적으로 작동하는지 확인한 baseline 검증 단계**이다.

다음 단계에서는 기존 영상처리 파이프라인의 출력:

```text
video_data.json
├── video_id
├── duration / fps / resolution
└── segments
    ├── segment_id
    ├── start_time
    ├── end_time
    ├── transcript
    └── frame_dir
```

을 Moment-DETR이 사용할 수 있는 feature 형태로 연결해야 한다.

예정 흐름:

```text
우리 영상
→ segment / frame
→ pretrained Visual Encoder
→ video feature
→ Moment-DETR
← 사용자 자연어 Query
→ start_time / end_time / score
```

이 단계가 성공하면 우리 서비스의 핵심 기능인 **문장형 검색 → 관련 영상 구간 반환**이 처음으로 연결된다.
